In [6]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# rendering the table for optimal analysis
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

# printing the first 5 rows of the dataset, to create a working set
df = pd.read_csv('../data/synthetic_fraud_data.csv', nrows=1000)

print(df.head())


  transaction_id customer_id       card_number                         timestamp merchant_category merchant_type        merchant     amount currency    country          city city_size        card_type  card_present   device channel                device_fingerprint       ip_address  distance_from_home  high_risk_merchant  transaction_hour  weekend_transaction                                                                                                                                        velocity_last_hour  is_fraud
0    TX_a0ad2a2a  CUST_72886  6646734767813109  2024-09-30 00:00:01.034820+00:00        Restaurant     fast_food       Taco Bell     294.87      GBP         UK  Unknown City    medium  Platinum Credit         False  iOS App  mobile  e8e6160445c935fd0001501e4cbac8bc   197.153.60.199                   0               False                 0                False  {'num_transactions': 1197, 'total_amount': 33498556.080464985, 'unique_merchants': 105, 'unique_countries': 12,

In [7]:
# Visualising the breakdown of fraud labels in the dataset 
print(df['is_fraud'].value_counts(normalize=True))
print(df['is_fraud'].value_counts())

is_fraud
False    0.739
True     0.261
Name: proportion, dtype: float64
is_fraud
False    739
True     261
Name: count, dtype: int64


In [8]:
# new dataframe created without unnecessary columns that do not influence final categorisation, prevents overfitting and speeds up the model
df_cleaned = df.drop(columns=['transaction_id', 'customer_id', 'card_number', 'city', 'city_size', 'card_present', 'ip_address', 'device_fingerprint'])

# old data frame is kept for final comparison 
print(df_cleaned.head())

                          timestamp merchant_category merchant_type        merchant     amount currency    country        card_type   device channel  distance_from_home  high_risk_merchant  transaction_hour  weekend_transaction                                                                                                                                        velocity_last_hour  is_fraud
0  2024-09-30 00:00:01.034820+00:00        Restaurant     fast_food       Taco Bell     294.87      GBP         UK  Platinum Credit  iOS App  mobile                   0               False                 0                False  {'num_transactions': 1197, 'total_amount': 33498556.080464985, 'unique_merchants': 105, 'unique_countries': 12, 'max_single_amount': 1925480.6324148502}     False
1  2024-09-30 00:00:01.764464+00:00     Entertainment        gaming           Steam    3368.97      BRL     Brazil  Platinum Credit     Edge     web                   1                True                 0          

In [9]:
print("Full DataFrame columns datatypes")
df.dtypes # identifying categorical columns datatype

Full DataFrame columns datatypes


transaction_id             str
customer_id                str
card_number              int64
timestamp                  str
merchant_category          str
merchant_type              str
merchant                   str
amount                 float64
currency                   str
country                    str
city                       str
city_size                  str
card_type                  str
card_present              bool
device                     str
channel                    str
device_fingerprint         str
ip_address                 str
distance_from_home       int64
high_risk_merchant        bool
transaction_hour         int64
weekend_transaction       bool
velocity_last_hour         str
is_fraud                  bool
dtype: object

In [10]:
import ast
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# coverting the booleans values into numerical binary values 
df_cleaned['is_fraud'] = df_cleaned['is_fraud'].astype(int)
df_cleaned['high_risk_merchant'] = df_cleaned['high_risk_merchant'].astype(int)
df_cleaned['weekend_transaction'] = df_cleaned['weekend_transaction'].astype(int)

# parsing meshed text into python dictornary data type, unpacking the JSON 
df_cleaned['parsed_velocity'] = df_cleaned['velocity_last_hour'].apply(ast.literal_eval)
# seperate all the dict keys into standalone columns
expanded_velocity = df_cleaned['parsed_velocity'].apply(pd.Series)
# merging the new column to the dataset
df_cleaned = pd.concat([df_cleaned, expanded_velocity], axis=1)
# dropping old meshed string columns and temporaty column 
df_cleaned = df_cleaned.drop(columns=['velocity_last_hour', 'parsed_velocity'])

# defining the categories to encode
categorical_columns = ['merchant_category', 'merchant_type', 'country', 'device', 'channel', 'currency']
numerical_columms = ['amount', 'distance_from_home', 'num_transactions', 'total_amount']

# Applying OneHotEncoder to the categorical columns, and StandardScaler to numerical columns
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(sparse_output=False), categorical_columns),
        ('num', StandardScaler(), numerical_columms)
    ],
    # preventing deletion of the unspecified columns
    remainder='passthrough' 
)

preprocessor.set_output(transform="pandas")

# running fit/transfrom transformation on the fully preprocessed dataset
df_processed = preprocessor.fit_transform(df_cleaned)

print(df_processed.head())

   cat__merchant_category_Education  cat__merchant_category_Entertainment  cat__merchant_category_Gas  cat__merchant_category_Grocery  cat__merchant_category_Healthcare  cat__merchant_category_Restaurant  cat__merchant_category_Retail  cat__merchant_category_Travel  cat__merchant_type_airlines  cat__merchant_type_booking  cat__merchant_type_casual  cat__merchant_type_events  cat__merchant_type_fast_food  cat__merchant_type_gaming  cat__merchant_type_hotels  cat__merchant_type_local  cat__merchant_type_major  cat__merchant_type_medical  cat__merchant_type_online  cat__merchant_type_pharmacy  cat__merchant_type_physical  cat__merchant_type_premium  cat__merchant_type_streaming  cat__merchant_type_supplies  cat__merchant_type_transport  cat__country_Australia  cat__country_Brazil  cat__country_Canada  cat__country_France  cat__country_Germany  cat__country_Japan  cat__country_Mexico  cat__country_Nigeria  cat__country_Russia  cat__country_Singapore  cat__country_UK  cat__country_USA  \
0 

In [16]:
from sklearn.model_selection import train_test_split

# seperating target feature from the rest of the dataset
# target label (y) seperate features (x)
x_dataframe = df_processed.drop(columns=['remainder__is_fraud'])

#print(x_dataframe)

#Extracting target column as y
y_dataframe = df_processed['remainder__is_fraud']

#print(y_dataframe)

# perfomring a Train-Test split: 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    x_dataframe,
    y_dataframe,
    test_size=0.2,
    random_state=24,
    stratify=y_dataframe
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"\nFraud count in y_train:\n{y_train.value_counts()}")

Training set shape: (800, 73)
Testing set shape: (200, 73)

Fraud count in y_train:
remainder__is_fraud
0    591
1    209
Name: count, dtype: int64
